# FINE TUNING TEMPLATE
*A simple template that this repository is expected to use. Modify if needed.*

## FINE-TUNING CODE FOR EXPERIMENT (EXPERIMENT)
(DESCRIPTION)

### Dependencies
Installed in the first cell:
- `unsloth[colab-new]` -- QLoRA training, Colab-specific build
- `xformers` -- attention optimization
- `trl` -- SFTTrainer
- `peft` -- LoRA adapters
- `accelerate` -- device placement
- `bitsandbytes` -- 4-bit quantization

### Dataset
(link dataset)

**Colab:** download the file, upload it when prompted by the cell.

**Local:** download the file, put it inside the root folder from which this notebook runs.

For local runs, replace `unsloth[colab-new]` with the correct CUDA-torch variant from the [Unsloth install docs](https://github.com/unslothai/unsloth), or install dependencies separately.

### DATA LOADING - COLAB ONLY. - run these cells if you are on Google Colab.

In [ ]:
# --- DEPENDENCIES (COLAB ONLY) ---
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# --- ALWAYS RUN - MODIFY IF NEEDED ---
"""
Saves model weights and checkpoints, so they are not lost on disconnect.

- COLAB: Connect Google Drive, so everything is saved in the pointed folder in cloud. WARNING: make sure you have enough space in your Google Drive.

- LOCAL: Automatically saves them at root folder.
"""
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = "/content/drive/MyDrive/{EXPERIMENT_NAME}" # <-- CHANGE HERE
except ImportError:
    OUTPUT_DIR = "{EXPERIMENT_NAME}" # <-- CHANGE HERE

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Checkpoints will save to:", OUTPUT_DIR)

In [ ]:
# --- COLAB ONLY - UPLOAD DATASET ---
from google.colab import files
uploaded = files.upload()  # select mixed_reasoning_v1.jsonl

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="mixed_reasoning_v1.jsonl", split="train")
print(dataset)

# Apply Qwen chat template
def format_row(row):
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False
    )}

dataset = dataset.map(format_row)
print(dataset[0]["text"][:500])

In [ ]:
# --- ATTACH LORA ADAPTERS ---
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from trl import SFTTrainer, SFTConfig

# parameters as described in the experiment writeup
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        dataset_text_field="text",
        max_seq_length=2048,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

In [ ]:
trainer_stats = trainer.train() # if you are loading a checkpoint - trainer.train(resume_from_checkpoint=True)

In [ ]:
# The final result is saved in your drive/notebook root directory in `/final_adapter`.

model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print("Saved to", f"{OUTPUT_DIR}/final_adapter")